In [2]:
# Cell 1: Setup and Imports
import json
import torch
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List, Dict, Tuple
import warnings
import time
from datetime import datetime
warnings.filterwarnings('ignore')

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
from sklearn.model_selection import train_test_split
import pickle

print("🔥 BERT NER Training Notebook")
print("=" * 50)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

🔥 BERT NER Training Notebook
Using device: cuda
GPU Name: GRID V100S-32Q


In [3]:
# Cell 2: Label Configuration (Same as RoBERTa)
BASE_LABELS = [
    'CHANGE', 'LOC', 'LULC', 'DATE', 'PERCENT', 
    'CARDINAL', 'COORDINATES', 'SURFACE_UNIT', 'PROCESS', 'QUANTITY'
]

# Create IOB2 labels
labels_list = ["O"]
for label in BASE_LABELS:
    labels_list.append(f"B-{label}")
    labels_list.append(f"I-{label}")

label2id = {label: i for i, label in enumerate(labels_list)}
id2label = {i: label for i, label in enumerate(labels_list)}

print(f"Total labels: {len(labels_list)}")
print("Label mapping created ✅")

Total labels: 21
Label mapping created ✅


In [4]:
# Cell 3: Load Data
DATA_FILE = 'merged_file.json2'  # Same data as RoBERTa

with open(DATA_FILE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"📊 Loaded {len(raw_data)} sentences")

# Quick data analysis
entity_counts = defaultdict(int)
for item in raw_data:
    for entity in item['entities']:
        entity_counts[entity['label']] += 1

print("\n📈 Entity distribution:")
for entity_type, count in sorted(entity_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {entity_type}: {count}")

📊 Loaded 14522 sentences

📈 Entity distribution:
  CHANGE: 10285
  LULC: 6728
  CARDINAL: 5652
  DATE: 4891
  LOC: 3959
  PERCENT: 1618
  COORDINATES: 1383
  PROCESS: 1282
  SURFACE_UNIT: 938
  QUANTITY: 131
  RESEARCH_TERM: 17


In [5]:
# Cell 4: BERT Tokenizer & Data Preparation
print("🤖 Initializing BERT...")

# BERT tokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def prepare_bert_dataset(data, tokenizer, label2id, max_length=512):
    """Prepare dataset for BERT training"""
    
    def process_example(example):
        sentence = example['original_sentence']
        entities = example['entities']
        
        # Tokenize
        encoding = tokenizer(
            sentence, 
            truncation=True, 
            max_length=max_length, 
            return_offsets_mapping=True, 
            padding='max_length'
        )
        
        offset_mapping = encoding['offset_mapping']
        labels = ['O'] * len(offset_mapping)
        
        # Process entities
        for entity in entities:
            start_char = entity['start_char']
            end_char = entity['end_char']
            entity_label = entity['label']
            
            if entity_label not in BASE_LABELS:
                continue
            
            # Find overlapping tokens
            entity_tokens = []
            for idx, (token_start, token_end) in enumerate(offset_mapping):
                if token_start == 0 and token_end == 0:
                    continue
                if token_start < end_char and token_end > start_char:
                    entity_tokens.append(idx)
            
            # Assign BIO labels
            if entity_tokens:
                labels[entity_tokens[0]] = f'B-{entity_label}'
                for idx in entity_tokens[1:]:
                    labels[idx] = f'I-{entity_label}'
        
        # Convert to IDs
        label_ids = [label2id.get(label, label2id['O']) for label in labels]
        
        return {
            'input_ids': encoding['input_ids'],
            'attention_mask': encoding['attention_mask'],
            'labels': label_ids,
            'original_sentence': sentence
        }
    
    processed_data = []
    for i, example in enumerate(data):
        if i % 1000 == 0:
            print(f"🔄 BERT Processing: {i}/{len(data)}")
        processed_data.append(process_example(example))
    
    return processed_data

# Process data
processed_data = prepare_bert_dataset(raw_data, bert_tokenizer, label2id)
print("✅ Data preprocessing completed")

🤖 Initializing BERT...
🔄 BERT Processing: 0/14522
🔄 BERT Processing: 1000/14522
🔄 BERT Processing: 2000/14522
🔄 BERT Processing: 3000/14522
🔄 BERT Processing: 4000/14522
🔄 BERT Processing: 5000/14522
🔄 BERT Processing: 6000/14522
🔄 BERT Processing: 7000/14522
🔄 BERT Processing: 8000/14522
🔄 BERT Processing: 9000/14522
🔄 BERT Processing: 10000/14522
🔄 BERT Processing: 11000/14522
🔄 BERT Processing: 12000/14522
🔄 BERT Processing: 13000/14522
🔄 BERT Processing: 14000/14522
✅ Data preprocessing completed


In [6]:
# Cell 5: Train/Test Split
print("✂️ Splitting data...")

# IMPORTANT: Use SAME random seed as RoBERTa for fair comparison
train_data, test_data = train_test_split(
    processed_data, 
    test_size=0.1, 
    random_state=42,  # SAME SEED!
    shuffle=True
)

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(f"📊 BERT Dataset Split:")
print(f"  Train: {len(train_data)} examples")
print(f"  Test: {len(test_data)} examples")

# Save dataset
dataset_dict.save_to_disk('./bert_processed_dataset')
with open('bert_label_mappings.pkl', 'wb') as f:
    pickle.dump({'label2id': label2id, 'id2label': id2label}, f)

print("💾 BERT dataset saved")

✂️ Splitting data...
📊 BERT Dataset Split:
  Train: 13069 examples
  Test: 1453 examples


Saving the dataset (0/1 shards):   0%|          | 0/13069 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1453 [00:00<?, ? examples/s]

💾 BERT dataset saved


In [7]:
# Cell 6: Model Setup
print("🏗️ Setting up BERT model...")

# Load BERT model
bert_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(labels_list),
    id2label=id2label,
    label2id=label2id
)

bert_model.to(device)

# Model info
total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)

print(f"📊 BERT Model Info:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / (1024**2):.1f} MB")

🏗️ Setting up BERT model...


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


📊 BERT Model Info:
  Total parameters: 108,907,797
  Trainable parameters: 108,907,797
  Model size: ~415.5 MB


In [8]:
# Cell 7: Evaluation Metrics (Same as RoBERTa)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    """Compute NER metrics with LULC focus"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    true_labels = []
    true_predictions = []
    
    for prediction, label in zip(predictions, labels):
        true_label = []
        true_prediction = []
        
        for pred_id, label_id in zip(prediction, label):
            if label_id != -100:  # Skip padding
                true_label.append(id2label[label_id])
                true_prediction.append(id2label[pred_id])
        
        true_labels.append(true_label)
        true_predictions.append(true_prediction)
    
    # Calculate metrics
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    
    # Extract LULC-specific metrics
    lulc_metrics = results.get('LULC', {})
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
        "lulc_precision": lulc_metrics.get('precision', 0),
        "lulc_recall": lulc_metrics.get('recall', 0),
        "lulc_f1": lulc_metrics.get('f1-score', 0),
    }

print("📏 Evaluation metrics configured")

📏 Evaluation metrics configured


In [9]:
# Cell 8: Training Configuration
# IDENTICAL to RoBERTa for fair comparison
training_args = TrainingArguments(
    output_dir="./bert_ner_model",
    learning_rate=3e-05,                    # Same as RoBERTa
    per_device_train_batch_size=8,         # Same as RoBERTa
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,         # Same as RoBERTa
    num_train_epochs=6,                    # Same as RoBERTa
    weight_decay=0.01,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",       # Focus on LULC
    greater_is_better=True,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    seed=42,                               # Same seed!
    report_to="none"
)

# Data collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=bert_tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

print("⚙️ BERT Training Configuration:")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  FP16: {training_args.fp16}")

⚙️ BERT Training Configuration:
  Learning rate: 3e-05
  Batch size: 8
  Epochs: 6
  FP16: True


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [10]:
# Cell 9: Training
print("🚀 Starting BERT Training...")

# Create trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=bert_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train with timing
start_time = time.time()
train_result = trainer.train()
training_time = time.time() - start_time

print("✅ BERT Training completed!")
print(f"⏱️ Training time: {training_time/60:.1f} minutes")
print(f"📉 Final training loss: {train_result.training_loss:.4f}")

🚀 Starting BERT Training...


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy,Lulc Precision,Lulc Recall,Lulc F1
500,0.007800,0.007005,0.804489,0.843015,0.823302,0.997992,0.954173,0.981481,0
1000,0.004300,0.005496,0.819638,0.884670,0.850913,0.998296,0.964052,0.993266,0
1500,0.003000,0.005072,0.851782,0.894021,0.872390,0.998427,0.962541,0.994949,0
2000,0.002200,0.004657,0.871590,0.896288,0.883766,0.998606,0.972039,0.994949,0


✅ BERT Training completed!
⏱️ Training time: 15.5 minutes
📉 Final training loss: 0.0500


In [11]:
# Cell 10: Evaluation
print("📊 Evaluating BERT model...")

eval_results = trainer.evaluate()

print("\n📈 BERT Evaluation Results:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print(f"\n⭐ BERT LULC Performance:")
print(f"  Precision: {eval_results.get('eval_lulc_precision', 0):.4f}")
print(f"  Recall: {eval_results.get('eval_lulc_recall', 0):.4f}")
print(f"  F1-Score: {eval_results.get('eval_lulc_f1', 0):.4f}")

📊 Evaluating BERT model...



📈 BERT Evaluation Results:
  eval_loss: 0.0047
  eval_precision: 0.8716
  eval_recall: 0.8963
  eval_f1: 0.8838
  eval_accuracy: 0.9986
  eval_lulc_precision: 0.9720
  eval_lulc_recall: 0.9949
  eval_runtime: 18.2770
  eval_samples_per_second: 79.4990
  eval_steps_per_second: 9.9580
  epoch: 5.9957

⭐ BERT LULC Performance:
  Precision: 0.9720
  Recall: 0.9949
  F1-Score: 0.0000


In [12]:
# Cell 11: Save Results for Comparison
print("💾 Saving BERT results for comparison...")

# Save model
trainer.save_model()
trainer.save_state()

# Save comprehensive results
bert_results = {
    'model_name': 'bert-base-uncased',
    'model_type': 'BERT',
    'training_time_seconds': training_time,
    'training_time_minutes': training_time / 60,
    'training_loss': train_result.training_loss,
    'eval_results': eval_results,
    'model_parameters': total_params,
    'model_size_mb': total_params * 4 / (1024**2),
    'training_args': training_args.to_dict(),
    'dataset_size': {
        'train': len(train_data),
        'test': len(test_data),
        'total': len(processed_data)
    },
    'timestamp': datetime.now().isoformat(),
    'labels': BASE_LABELS,
    'label_mappings': {'label2id': label2id, 'id2label': id2label}
}

# Save to JSON for comparison
with open('bert_training_results.json', 'w') as f:
    json.dump(bert_results, f, indent=2, default=str)

# Save to pickle for easy loading
with open('bert_results.pkl', 'wb') as f:
    pickle.dump(bert_results, f)

print("✅ BERT results saved:")
print("  • bert_training_results.json")
print("  • bert_results.pkl")
print("  • ./bert_ner_model/ (model files)")

print(f"\n🎯 BERT Summary:")
print(f"  LULC F1: {eval_results.get('eval_lulc_f1', 0):.4f}")
print(f"  Overall F1: {eval_results.get('eval_f1', 0):.4f}")
print(f"  Training time: {training_time/60:.1f} minutes")
print(f"  Model size: {total_params/1_000_000:.1f}M parameters")

💾 Saving BERT results for comparison...
✅ BERT results saved:
  • bert_training_results.json
  • bert_results.pkl
  • ./bert_ner_model/ (model files)

🎯 BERT Summary:
  LULC F1: 0.0000
  Overall F1: 0.8838
  Training time: 15.5 minutes
  Model size: 108.9M parameters


In [16]:
# Cell 12: Quick Test
def test_bert_model(text):
    """Quick test function"""
    inputs = bert_tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = bert_model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)
    
    tokens = bert_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    labels = [id2label[pred.item()] for pred in predictions[0]]
    
    # Extract entities
    entities = []
    current_entity = None
    
    for token, label in zip(tokens, labels):
        if token in bert_tokenizer.all_special_tokens:
            continue
            
        if label.startswith('B-'):
            if current_entity:
                entities.append(current_entity)
            current_entity = {'text': token, 'label': label[2:]}
        elif label.startswith('I-') and current_entity and label[2:] == current_entity['label']:
            current_entity['text'] += f' {token}'
        else:
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    if current_entity:
        entities.append(current_entity)
    
    return entities

# Test examples
test_sentences = [
    "In the Amazon Basin, Nashe Watershed  deforestation has led to the loss of nearly 20% of the rainforest since 1970, amounting to roughly 800,000 km² of forest removed.",
    "Between 2000 and 2010, forest cover in the Dry Chaco region (Argentina and Paraguay) decreased by about 20% (approximately 9.5 million hectares) due to agricultural expansion.",
    "Between 2000 and 2020, cropland in East Africa expanded significantly while forest cover shrank by about 9%, highlighting extensive conversion of forests to agricultural land.",
    "Agricultural land decreased by 30% while built-up area increased significantly.",
    "Urban sprawl led to the conversion of green spaces into commercial zones.",
    "The development of industrial parks replaced former wetlands and grasslands and built-up area.",
    "In Costa Rica, forest cover increased from around 24% in 1985 to over 50% by 2011 thanks to reforestation policies.",
    "In Shanghai, China, urban area expanded from 308 km² in 1984 to about 1,300 km ² by 2014.",
    "Mangrove forests declined by approximately 30% in Southeast Asia since 1980 due to shrimp farming.",
    "In Central Asia, the Aral Sea shrank from 68,000 km² in 1960 to 6,800 km² by 2010, a 90% decrease.",
    "In Brazil’s Cerrado, cropland increased by 120% since 2000, converting natural savannas.",
    "Singapore increased its land area by 22% from the 1960s to 2015 via coastal reclamation.",
    "Borneo lost roughly 30% of its old-growth rainforest between 1973 and 2015 due to logging.",
    "Forest area in Spain rose from 28% in 1990 to 37% in 2015 due to land abandonment.",
    "In Zambia, 5,000 km² of Miombo woodland was cleared between 2000 and 2015 for agriculture.",
    "Russia saw about 30 million hectares of farmland abandoned post-1991, allowing reforestation.",
    "In India, urban land grew by 250% from 1991 to 2018, converting 100,000 hectares of cropland.",
    "Forest loss in Sumatra, Indonesia exceeded 20,000 km² between 1990 and 2010 for palm oil.",
    "Between 2013 and 2018, 1 million hectares of forest were cleared in Queensland, Australia.",
    "Forest cover in Nepal grew from 26% in the early 1990s to 45% by 2016 through community forestry.",
    "The Gobi Desert expanded by 3,600 km² from 1994 to 2018, replacing grassland in China.",
    "Las Vegas urbanized over 130% more land between 1992 and 2018, replacing desert with housing.",
    "Coastal erosion processes intensified land loss along the Louisiana coastline since 1950.",
    "The Gangotri Glacier in India lost 5% of its mass from 1980 to 2015 due to climate change.",
    "Forests in Europe expanded by 14 million hectares from 1990 to 2020 due to regrowth.",
    "Thailand saw a 12% increase in agricultural land between 2001 and 2015 near the Mekong basin.",
    "Between 2000 and 2015, urban areas in Sub-Saharan Africa expanded by over 100,000 hectares.",
    "Forest degradation in Cameroon resulted in a 15% canopy density loss between 2002 and 2020.",
    "Land reclamation in the Netherlands added 1,600 km² of polders since the 1950s.",
    "Reforestation efforts in Ethiopia led to a 10% increase in forest area from 2000 to 2020."
]

print("\n🧪 Testing BERT model:")
for sentence in test_sentences:
    print(f"\n📝 Input: {sentence}")
    entities = test_bert_model(sentence)
    print("🏷️ BERT detected:")
    for entity in entities:
        print(f"   • {entity['text']} → {entity['label']}")

print("\n🎉 BERT notebook completed!")
print("Ready for comparison with RoBERTa! 🤖 vs 🤖")


🧪 Testing BERT model:

📝 Input: In the Amazon Basin, Nashe Watershed  deforestation has led to the loss of nearly 20% of the rainforest since 1970, amounting to roughly 800,000 km² of forest removed.
🏷️ BERT detected:
   • def ##orestation → PROCESS
   • loss → CHANGE
   • nearly 20 % → PERCENT
   • 1970 → DATE
   • roughly → CARDINAL
   • forest → LULC

📝 Input: Between 2000 and 2010, forest cover in the Dry Chaco region (Argentina and Paraguay) decreased by about 20% (approximately 9.5 million hectares) due to agricultural expansion.
🏷️ BERT detected:
   • between 2000 and 2010 → DATE
   • forest → LULC
   • argentina → LOC
   • paraguay → LOC
   • decreased → CHANGE
   • about 20 % → PERCENT
   • 9 . 5 million → COORDINATES
   • expansion → CHANGE

📝 Input: Between 2000 and 2020, cropland in East Africa expanded significantly while forest cover shrank by about 9%, highlighting extensive conversion of forests to agricultural land.
🏷️ BERT detected:
   • between 2000 and 2020 → DATE


In [15]:
# First, make sure tqdm is installed
!pip install tqdm -q

from tqdm import tqdm

def extract_all_lulc_terms(texts, model_path="./bert_ner_model"):
    """
    Extract all unique LULC terms from a list of texts.
    This gives you the comprehensive list you requested.
    """
    all_lulc_terms = []
    
    # Add progress bar for processing texts
    print("🔄 Extracting entities from texts...")
    for text in tqdm(texts, desc="Processing sentences", unit="sentence"):
        entities = extract_entities(text, model_path)
        lulc_entities = [ent['text'].lower() for ent in entities if ent['label'] == 'LULC']
        all_lulc_terms.extend(lulc_entities)
    
    # Get unique terms
    unique_lulc_terms = sorted(list(set(all_lulc_terms)))
    return unique_lulc_terms

# Extract LULC terms from your original data
all_texts = [item['original_sentence'] for item in raw_data]
print(f"📊 Total sentences to process: {len(all_texts)}")

unique_lulc_terms = extract_all_lulc_terms(all_texts)

print(f"\n✅ Found {len(unique_lulc_terms)} unique LULC terms in your data:\n")

# Show first 20 terms
print("First 20 LULC terms:")
for i, term in enumerate(unique_lulc_terms[:20], 1):
    print(f"  {i}. {term}")

if len(unique_lulc_terms) > 20:
    print(f"  ... and {len(unique_lulc_terms) - 20} more terms")

# Save to file
with open('unique_lulc_terms.json', 'w') as f:
    json.dump(unique_lulc_terms, f, indent=2)
print(f"\n💾 Saved all {len(unique_lulc_terms)} terms to 'unique_lulc_terms.json'")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


📊 Total sentences to process: 14522
🔄 Extracting entities from texts...


Processing sentences:   0%|                                                                          | 0/14522 [00:00<?, ?sentence/s]


NameError: name 'extract_entities' is not defined